# 🎯 Polymarket Trade Monitor
Watches elite wallets and alerts instantly when they enter a new position.

**Shows:** Market name · YES/NO · Price · Implied probability · Size

## 1. Keep Colab Alive

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

## 2. Install & Imports

In [ ]:
import requests, time, json, os
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML, clear_output

print('✅ Ready')

## 3. Wallet List & Config

In [ ]:
POLL_INTERVAL = 60
MIN_POSITION_SIZE = 10
DATA_API = "https://data-api.polymarket.com/v1"
EXCEL_FILE = "polymarket_trades_log.xlsx"

WATCH_WALLETS = [
    ("Car",              "0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b", "POLITICS+TECH",     295380),
    ("bobe2",            "0xed107a85a4585a381e48c7f7ca4144909e7dd2e5", "ECONOMICS+POLITICS", 290759),
    ("cry.eth2",         "0xe3726a1b9c6ba2f06585d1c9e01d00afaedaeb38", "FINANCE+POLITICS",  191506),
    ("anoin123",         "0x96489abcb9f583d6834f8abc99ef56eca1de7bdc", "POLITICS+TECH",     144555),
    ("HorizonSplendidView", "0x0a2227b278c04da9962fcf96490e17f3dfb9bc1", "SPORTS",          4016108),
    ("reachingthesky",      "0xefbc5fec8d7b0acd c8911bd9a98d96348f9a2",  "SPORTS",          3742635),
    ("beachboy4",           "0xc2e7800b5af46e693872b17b7a5e7f0563be51",  "SPORTS",          3058057),
    ("majorexploiter",      "0x019782cab5d844f02bafb71f512758e78579f3c", "SPORTS",          2416975),
    ("sovereign2013",       "0xee613b5f1c3ee4f4ad9a9c5f3e2da107fe3e2033","SPORTS",         1938055),
    ("CemeterySun",         "0xb2bf06bd5abb38e6688840d6c0f9eee0b4c1efb6","SPORTS",         1927133),
    ("surfandturf",         "0x9f2fe025f84839ca81dd0338889c5d2ca4984840","SPORTS",         1882357),
    ("denizz",              "0xbaa2bcb5439e985ce4ccf815b4700027d1b92c73","POLITICS",         983344),
    ("SecondWindCapital",   "0x8c80d213c0cbad777d06ee3f58f6ca4bc03102c3","POLITICS",         438637),
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json",
}

session = requests.Session()
session.headers.update(HEADERS)

print(f'✅ Watching {len(WATCH_WALLETS)} wallets')
print(f'   Poll interval: every {POLL_INTERVAL}s')
print(f'   Min position size: ${MIN_POSITION_SIZE}')

## 4. Core Functions

In [ ]:
def get_activity(wallet, limit=20):
    try:
        res = session.get(
            f"{DATA_API}/activity",
            params={"user": wallet, "limit": limit},
            timeout=15
        )
        return res.json() if res.status_code == 200 else []
    except:
        return []

def get_positions(wallet):
    try:
        res = session.get(
            f"{DATA_API}/positions",
            params={"user": wallet, "limit": 100},
            timeout=15
        )
        return res.json() if res.status_code == 200 else []
    except:
        return []

def parse_trade(activity_item):
    try:
        price = float(activity_item.get("price", 0))
        size  = float(activity_item.get("size", 0))
        return {
            "market":    activity_item.get("title") or activity_item.get("market", "Unknown"),
            "outcome":   activity_item.get("outcome", "?"),
            "side":      activity_item.get("side", "?"),
            "price":     price,
            "prob_pct":  round(price * 100, 1),
            "size":      size,
            "usd_value": round(price * size, 2),
            "tx_hash":   activity_item.get("transactionHash", ""),
            "timestamp": activity_item.get("timestamp", 0),
        }
    except:
        return None

def log_to_excel(trade_data):
    df = pd.DataFrame([trade_data])
    if os.path.exists(EXCEL_FILE):
        existing_df = pd.read_excel(EXCEL_FILE)
        df = pd.concat([existing_df, df], ignore_index=True)
    df.to_excel(EXCEL_FILE, index=False)

def format_alert(username, category, pnl, trade):
    side = trade['side']
    outcome = trade['outcome']
    
    if side == 'BUY':
        side_color = '#28a745'
        side_icon  = '🟢 BUY'
    else:
        side_color = '#dc3545'
        side_icon  = '🔴 SELL'

    prob = trade['prob_pct']
    if prob >= 70:
        prob_color = '#28a745'
    elif prob >= 40:
        prob_color = '#fd7e14'
    else:
        prob_color = '#dc3545'

    ts = datetime.fromtimestamp(trade['timestamp']).strftime('%H:%M:%S') if trade['timestamp'] else '?'
    polymarket_url = f"https://polymarket.com/profile/{username}"

    return f"""
    <div style="
        border: 2px solid {side_color};
        border-radius: 10px;
        padding: 16px 20px;
        margin: 10px 0;
        background: #1a1a2e;
        font-family: Arial, sans-serif;
        color: #ffffff;
    ">
        <div style="display:flex; justify-content:space-between; align-items:center;">
            <span style="font-size:18px; font-weight:bold; color:{side_color}">
                🚨 NEW TRADE DETECTED
            </span>
            <span style="font-size:12px; color:#aaa">{ts}</span>
        </div>

        <div style="margin:10px 0; padding:10px; background:#16213e; border-radius:6px;">
            <span style="color:#aaa; font-size:12px;">TRADER</span><br>
            <span style="font-size:16px; font-weight:bold;">👤 {username}</span>
            <span style="margin-left:10px; color:#aaa; font-size:12px;">{category} · ${pnl:,.0f} PnL this month</span>
        </div>

        <div style="margin:10px 0; padding:10px; background:#16213e; border-radius:6px;">
            <span style="color:#aaa; font-size:12px;">MARKET</span><br>
            <span style="font-size:15px;">📋 {trade['market'][:80]}</span>
        </div>

        <div style="display:grid; grid-template-columns:1fr 1fr 1fr 1fr; gap:10px; margin-top:12px;">
            <div style="background:#16213e; padding:10px; border-radius:6px; text-align:center;">
                <div style="color:#aaa; font-size:11px;">ACTION</div>
                <div style="font-size:16px; font-weight:bold; color:{side_color};">{side_icon}</div>
            </div>
            <div style="background:#16213e; padding:10px; border-radius:6px; text-align:center;">
                <div style="color:#aaa; font-size:11px;">OUTCOME</div>
                <div style="font-size:16px; font-weight:bold;">{'✅' if outcome.upper() in ['YES','1'] else '❌' if outcome.upper() in ['NO','0'] else '🎯'} {outcome}</div>
            </div>
            <div style="background:#16213e; padding:10px; border-radius:6px; text-align:center;">
                <div style="color:#aaa; font-size:11px;">PRICE / PROB</div>
                <div style="font-size:16px; font-weight:bold; color:{prob_color}">{trade['price']:.2f} ({prob}%)</div>
            </div>
            <div style="background:#16213e; padding:10px; border-radius:6px; text-align:center;">
                <div style="color:#aaa; font-size:11px;">SIZE</div>
                <div style="font-size:16px; font-weight:bold;">{trade['size']:.0f} shares</div>
            </div>
        </div>

        <div style="margin-top:12px; font-size:12px; color:#aaa;">
            💰 Approx value: <b style="color:#fff">${trade['usd_value']:,.2f}</b> &nbsp;&nbsp;
            🔗 <a href="{polymarket_url}" style="color:#4fc3f7;">View trader profile</a>
        </div>
    </div>
    """

print('✅ Functions loaded')

## 5. Initialize — Take First Snapshot

In [ ]:
seen_trades = {}
wallet_info = {}

print('📸 Taking initial snapshot...')
print('=' * 50)

for username, wallet, category, pnl in WATCH_WALLETS:
    wallet_info[wallet] = (username, category, pnl)
    activity = get_activity(wallet, limit=50)
    
    hashes = set()
    for item in activity:
        tx = item.get("transactionHash", "")
        if tx:
            hashes.add(tx)
    
    seen_trades[wallet] = hashes
    
    positions = get_positions(wallet)
    pos_count = len(positions) if isinstance(positions, list) else 0
    print(f'  ✓ {username:<25} | {len(hashes):>3} recent trades | {pos_count} open positions')
    time.sleep(0.3)

print('=' * 50)
print(f'✅ Snapshot complete. Now run Cell 6 to start monitoring.')

## 6. 🚀 Start Monitor

In [ ]:
alerts_log = []
cycle = 0

print('🟢 Monitor started. Watching', len(WATCH_WALLETS), 'wallets...')
print('   Stop with ⏹ button\n')

while True:
    cycle += 1
    now = datetime.utcnow().strftime('%H:%M:%S')
    new_found = 0

    for username, wallet, category, pnl in WATCH_WALLETS:
        activity = get_activity(wallet, limit=20)
        
        for item in activity:
            tx = item.get("transactionHash", "")
            side = item.get("side", "")
            
            if not tx or tx in seen_trades.get(wallet, set()):
                continue
            if side != "BUY":
                seen_trades[wallet].add(tx)
                continue
            
            trade = parse_trade(item)
            if not trade or trade["usd_value"] < MIN_POSITION_SIZE:
                seen_trades[wallet].add(tx)
                continue
            
            seen_trades[wallet].add(tx)
            alerts_log.append({"time": now, "username": username, "trade": trade})
            new_found += 1
            
            excel_data = {
                "Time (UTC)": now,
                "Username": username,
                "Market": trade["market"],
                "Action": side,
                "Outcome": trade["outcome"],
                "Price": trade["price"],
                "Probability (%)": trade["prob_pct"],
                "Size (Shares)": trade["size"],
                "USD Value": trade["usd_value"],
                "Tx Hash": tx
            }
            log_to_excel(excel_data)
            
            html = format_alert(username, category, pnl, trade)
            display(HTML(html))
        
        time.sleep(0.3)

    status = f'[Cycle {cycle} | {now} UTC] Checked {len(WATCH_WALLETS)} wallets'
    if new_found > 0:
        status += f' | ⚡ {new_found} NEW TRADE(S) ABOVE'
    else:
        status += f' | No new trades | Total alerts: {len(alerts_log)}'
    
    print(f'\r{status}', end='', flush=True)
    time.sleep(POLL_INTERVAL)

## 7. View Alert History

In [ ]:
if not alerts_log:
    print('No alerts yet.')
else:
    print(f'Total alerts this session: {len(alerts_log)}\n')
    for a in alerts_log:
        t = a['trade']
        print(f"[{a['time']}] {a['username']:<20} | {t['side']:<4} {t['outcome']:<6} @ {t['price']:.2f} ({t['prob_pct']}%) | ${t['usd_value']:>8,.2f} | {t['market'][:50]}")